# Phase 1: Dataset Analysis — Quantitative Forensics & Distributions

## Objective
Perform detailed statistical analysis of query lengths, passage lengths, answer presence, missing values, duplicates, and relevance distribution in `ai4bharat/MSMARCO-XI` using Parquet file splits.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_split_statistics(dataset_split):
    queries_len_words = []
    queries_len_chars = []
    passages_len_words = []
    passages_len_chars = []
    selected_counts = []
    missing_answers = 0
    missing_queries = 0
    query_ids = set()
    duplicates = 0
    
    for sample in dataset_split:
        qid = sample['query_id']
        if qid in query_ids:
            duplicates += 1
        else:
            query_ids.add(qid)
            
        q_text = sample.get('query', '')
        if not q_text:
            missing_queries += 1
        queries_len_words.append(len(q_text.split()))
        queries_len_chars.append(len(q_text))
        
        ans_text = sample.get('Answer', '')
        if not ans_text or str(ans_text).strip() == '':
            missing_answers += 1
            
        passages = sample.get('passages', {})
        trans_passages = passages.get('Translated_passages', [])
        is_sel = passages.get('is_selected', [])
        
        selected_counts.append(sum(is_sel))
        for p in trans_passages:
            passages_len_words.append(len(p.split()))
            passages_len_chars.append(len(p))
            
    stats = {
        'total_records': len(dataset_split),
        'unique_query_ids': len(query_ids),
        'duplicate_queries': duplicates,
        'missing_queries': missing_queries,
        'missing_answers': missing_answers,
        'query_len_words_mean': np.mean(queries_len_words),
        'query_len_words_p50': np.percentile(queries_len_words, 50),
        'query_len_words_p95': np.percentile(queries_len_words, 95),
        'passage_len_words_mean': np.mean(passages_len_words),
        'passage_len_words_p50': np.percentile(passages_len_words, 50),
        'passage_len_words_p95': np.percentile(passages_len_words, 95),
        'avg_selected_passages_per_query': np.mean(selected_counts)
    }
    return stats

print("Statistical Analysis Helper Defined Successfully.")

Statistical Analysis Helper Defined Successfully.


In [3]:
from datasets import load_dataset
try:
    hi_val = load_dataset("ai4bharat/MSMARCO-XI", data_files="validation/hinval.parquet", split="train")
    stats = analyze_split_statistics(hi_val)
    print("=== HINDI VALIDATION SPLIT METRICS ===")
    for k, v in stats.items():
        print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")
except Exception as e:
    print(f"Dataset loading exception: {e}")

Repo card metadata block was not found. Setting CardData to empty.


validation/hinval.parquet: reconstructing file:   0%|          |  0.00B /  462MB            

validation/hinval.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

=== HINDI VALIDATION SPLIT METRICS ===
  total_records: 97941
  unique_query_ids: 97940
  duplicate_queries: 1
  missing_queries: 0
  missing_answers: 0
  query_len_words_mean: 8.35
  query_len_words_p50: 7.00
  query_len_words_p95: 13.00
  passage_len_words_mean: 61.44
  passage_len_words_p50: 55.00
  passage_len_words_p95: 107.00
  avg_selected_passages_per_query: 0.59


## Key Takeaways for Chunking & Retrieval

- **Passage Length Profile**: Mean passage length is ~55-65 words (~300-400 characters). This matches single paragraph units.
- **Query Length Profile**: Average query length is ~6-9 words. Voice inputs map naturally to short-to-medium length queries.